# Indonesian stocks grouped by industry and sorted by market capitalization

> **Educational research universe—not a buy list.** This notebook places every stock with available company metadata into a separate industry table and sorts each table from the largest to the smallest provider-reported market capitalization.

The classifications are **Yahoo Finance industries, not official IDX-IC industries**. Market capitalization, shares outstanding, float shares, and classifications are provider metadata that can be stale or incorrect. Verify important figures against dated issuer and IDX disclosures before making a decision.

The source snapshots are dated 2026-09-22. Bulk Yahoo data remain in the Git-ignored `private/` directory because Yahoo data rights are separate from the open-source `yfinance` client.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 200)

# Objective: locate repository artifacts when executed from the root or notebook directory.
def locate(relative_path, private=False):
    candidates = [Path(relative_path), Path('products/stocks') / relative_path]
    if private:
        candidates.extend([
            Path('../../private') / relative_path,
            Path('private') / relative_path,
        ])
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not locate {relative_path}; tried {candidates}')

snapshot_date = '2026-09-22'
registry_path = locate(f'data/processed/ksei_registered_share_securities_{snapshot_date}.csv')
registry_manifest_path = locate(f'data/manifests/ksei_stock_universe_{snapshot_date}.json')
price_path = locate(f'market_data/yahoo/idx_latest_prices_{snapshot_date}.csv', private=True)
price_manifest_path = locate(f'market_data/yahoo/idx_latest_prices_{snapshot_date}.json', private=True)
metadata_path = locate(f'market_data/yahoo/idx_company_metadata_{snapshot_date}.csv', private=True)
metadata_manifest_path = locate(f'market_data/yahoo/idx_company_metadata_{snapshot_date}.json', private=True)

registry = pd.read_csv(registry_path)
registry_manifest = json.loads(registry_manifest_path.read_text())
prices = pd.read_csv(price_path)
price_manifest = json.loads(price_manifest_path.read_text())
metadata = pd.read_csv(metadata_path)
metadata_manifest = json.loads(metadata_manifest_path.read_text())

len(registry), len(prices), len(metadata)

(984, 980, 919)

## Coverage and definitions

The grouping universe consists of KSEI four-letter ticker candidates for which Yahoo returned company metadata. Stocks without Yahoo metadata cannot be assigned to a Yahoo industry and are reported as coverage exceptions rather than silently inserted into an invented category.

An “industry” below is the provider's descriptive industry field. It is more granular than its sector field. The five metadata-covered stocks with no industry value are retained under **Unclassified by Yahoo**.

In [2]:
coverage = pd.Series({
    'KSEI registered share securities': len(registry),
    'standard four-letter ticker candidates': int(registry['ticker'].astype(str).str.fullmatch(r'[A-Z]{4}').sum()),
    'Yahoo price candidates requested': int(price_manifest['requested_ticker_count']),
    'Yahoo daily prices available': int(prices['price_status'].eq('available').sum()),
    'Yahoo daily prices missing': int(prices['price_status'].ne('available').sum()),
    'Yahoo company metadata rows available': int(metadata['metadata_status'].eq('available').sum()),
    'stocks with a named Yahoo industry': int(metadata['industry'].notna().sum()),
    'stocks unclassified by Yahoo': int(metadata['industry'].isna().sum()),
    'named Yahoo industries': int(metadata['industry'].nunique(dropna=True)),
    'metadata rows with market capitalization': int(metadata['marketCap'].notna().sum()),
    'snapshot date': snapshot_date,
    'Yahoo metadata retrieval UTC': metadata_manifest['retrieved_at_utc'],
}, name='value').to_frame()
coverage

,value
KSEI registered share securities,984
standard four-letter ticker candidates,980
Yahoo price candidates requested,980
Yahoo daily prices available,919
Yahoo daily prices missing,61
Yahoo company metadata rows available,919
stocks with a named Yahoo industry,914
stocks unclassified by Yahoo,5
named Yahoo industries,112
metadata rows with market capitalization,919


## Build and validate the industry universe

In [3]:
# Objective: merge the dated registry, price, and company snapshots without duplicating tickers.
universe = metadata.loc[metadata['metadata_status'].eq('available')].copy()
universe = universe.merge(
    registry[['ticker', 'registry_name', 'detail_url']],
    on='ticker',
    how='left',
    validate='one_to_one',
)
universe = universe.merge(
    prices.loc[prices['price_status'].eq('available'), [
        'ticker', 'price_date', 'close_idr', 'volume_shares', 'price_status'
    ]],
    on='ticker',
    how='left',
    validate='one_to_one',
)

universe['industry_group'] = universe['industry'].fillna('Unclassified by Yahoo')
universe['company_name'] = (
    universe['longName']
    .fillna(universe['shortName'])
    .fillna(universe['registry_name'])
)
universe['market_cap_idr_billion'] = universe['marketCap'] / 1e9
universe['market_cap_idr_trillion'] = universe['marketCap'] / 1e12
universe['float_pct_of_outstanding'] = 100 * universe['floatShares'] / universe['sharesOutstanding']

assert universe['ticker'].is_unique
assert len(universe) == int(metadata['metadata_status'].eq('available').sum())
assert universe['marketCap'].notna().all()
assert universe['industry_group'].notna().all()
universe.shape

(919, 32)

## Industry catalogue

The catalogue is sorted by aggregate provider-reported market capitalization. This ordering is only for navigation; it does not rank industry attractiveness.

In [4]:
industry_summary = (
    universe.groupby('industry_group', dropna=False)
    .agg(
        stock_count=('ticker', 'size'),
        sector_count=('sector', 'nunique'),
        total_market_cap_idr=('marketCap', 'sum'),
        median_market_cap_idr=('marketCap', 'median'),
        price_coverage=('close_idr', 'count'),
        issued_shares_coverage=('sharesOutstanding', 'count'),
        float_shares_coverage=('floatShares', 'count'),
    )
    .sort_values(['total_market_cap_idr', 'stock_count'], ascending=[False, False])
)
industry_summary['total_market_cap_idr_trillion'] = industry_summary['total_market_cap_idr'] / 1e12
industry_summary['median_market_cap_idr_billion'] = industry_summary['median_market_cap_idr'] / 1e9
industry_summary[[
    'stock_count', 'sector_count', 'total_market_cap_idr_trillion',
    'median_market_cap_idr_billion', 'price_coverage',
    'issued_shares_coverage', 'float_shares_coverage'
]]

,stock_count,sector_count,total_market_cap_idr_trillion,median_market_cap_idr_billion,price_coverage,issued_shares_coverage,float_shares_coverage
industry_group,,,,,,,
Banks - Regional,49,1,2400.434540,7732.981137,49,49,49
Thermal Coal,33,1,1165.755143,8749.999718,33,33,33
Telecom Services,20,1,860.568660,6056.528839,20,20,20
Real Estate Services,30,1,655.390975,2526.682808,30,30,30
Other Industrial Metals & Mining,21,1,554.695186,4721.782292,21,21,21
Utilities - Renewable,7,1,473.283441,1738.514498,7,7,7
Farm Products,40,1,388.652734,2819.732668,40,40,40
Other Precious Metals & Mining,1,1,349.027833,349027.832955,1,1,1
Real Estate - Development,54,1,342.665546,969.062023,54,54,53


## Create one market-cap-sorted table for every industry

In [5]:
# Objective: create complete, non-overlapping industry tables sorted by market cap and ticker.
display_columns = [
    'rank_in_industry', 'ticker', 'company_name', 'sector', 'industry_group',
    'price_date', 'close_idr', 'market_cap_idr_billion',
    'market_cap_idr_trillion', 'sharesOutstanding', 'floatShares',
    'float_pct_of_outstanding', 'volume_shares', 'detail_url'
]

industry_order = industry_summary.index.tolist()
industry_tables = {}
for industry_name in industry_order:
    table = (
        universe.loc[universe['industry_group'].eq(industry_name)]
        .sort_values(['marketCap', 'ticker'], ascending=[False, True], na_position='last')
        .copy()
    )
    table.insert(0, 'rank_in_industry', range(1, len(table) + 1))
    industry_tables[industry_name] = table[display_columns].reset_index(drop=True)

combined_tickers = [
    ticker
    for table in industry_tables.values()
    for ticker in table['ticker'].tolist()
]
assert len(combined_tickers) == len(universe)
assert len(set(combined_tickers)) == len(universe)
assert set(combined_tickers) == set(universe['ticker'])
assert all(
    table['market_cap_idr_billion'].is_monotonic_decreasing
    for table in industry_tables.values()
)

validation = pd.Series({
    'industry tables created': len(industry_tables),
    'stocks across all tables': len(combined_tickers),
    'unique stocks across all tables': len(set(combined_tickers)),
    'stocks in source universe': len(universe),
    'duplicate ticker assignments': len(combined_tickers) - len(set(combined_tickers)),
    'all tables market-cap sorted descending': all(
        table['market_cap_idr_billion'].is_monotonic_decreasing
        for table in industry_tables.values()
    ),
}, name='value').to_frame()
validation

,value
industry tables created,113
stocks across all tables,919
unique stocks across all tables,919
stocks in source universe,919
duplicate ticker assignments,0
all tables market-cap sorted descending,True


## All industry tables

Within every table, rank 1 has the largest provider-reported market capitalization in that industry. Market capitalization is shown in both **Rp billion** and **Rp trillion**. `float_pct_of_outstanding` is calculated from Yahoo's reported float shares and shares outstanding; it is not a substitute for an official IDX free-float disclosure.

In [6]:
# Objective: display every industry as a visually separate, complete table.
for industry_name, table in industry_tables.items():
    industry_market_cap = table['market_cap_idr_trillion'].sum()
    display(Markdown(
        f'### {industry_name}  '
        f'\n{len(table):,} stocks; combined reported market cap: '
        f'Rp{industry_market_cap:,.2f} trillion'
    ))
    formatted = table.style.format({
        'close_idr': '{:,.0f}',
        'market_cap_idr_billion': '{:,.2f}',
        'market_cap_idr_trillion': '{:,.4f}',
        'sharesOutstanding': '{:,.0f}',
        'floatShares': '{:,.0f}',
        'float_pct_of_outstanding': '{:,.2f}%',
        'volume_shares': '{:,.0f}',
    }, na_rep='—')
    display(formatted)

### Banks - Regional  
49 stocks; combined reported market cap: Rp2,400.43 trillion

,rank_in_industry,ticker,company_name,sector,industry_group,price_date,close_idr,market_cap_idr_billion,market_cap_idr_trillion,sharesOutstanding,floatShares,float_pct_of_outstanding,volume_shares,detail_url
0,1,BBCA,PT Bank Central Asia Tbk,Financial Services,Banks - Regional,2026-09-22,"6,250","767,760.97",767.7610,"122,841,751,300","48,199,417,958",39.24%,"15,300,300",https://web.ksei.co.id/services/registered-securities/shares/lc/BBCA
1,2,BBRI,PT Bank Rakyat Indonesia (Persero) Tbk,Financial Services,Banks - Regional,2026-09-22,"3,260","488,216.78",488.2168,"149,878,079,535","69,249,932,878",46.20%,"37,461,100",https://web.ksei.co.id/services/registered-securities/shares/lc/BBRI
2,3,BMRI,PT Bank Mandiri (Persero) Tbk,Financial Services,Banks - Regional,2026-09-22,"4,210","391,417.75",391.4178,"92,487,999,999","37,719,733,333",40.78%,"24,686,000",https://web.ksei.co.id/services/registered-securities/shares/lc/BMRI
3,4,BBNI,PT Bank Negara Indonesia (Persero) Tbk,Financial Services,Banks - Regional,2026-09-22,"3,630","134,869.61",134.8696,"578,683,733","225,964,424",39.05%,"4,261,600",https://web.ksei.co.id/services/registered-securities/shares/lc/BBNI
4,5,BNLI,PT Bank Permata Tbk,Financial Services,Banks - Regional,2026-09-22,"2,140","75,980.76",75.9808,"26,880,234","26,412,358",98.26%,"2,600",https://web.ksei.co.id/services/registered-securities/shares/lc/BNLI
5,6,BRIS,PT Bank Syariah Indonesia Tbk,Financial Services,Banks - Regional,2026-09-22,"1,595","73,345.52",73.3455,"46,129,260,138","4,568,180,631",9.90%,"2,429,200",https://web.ksei.co.id/services/registered-securities/shares/lc/BRIS
6,7,MEGA,PT Bank Mega Tbk,Financial Services,Banks - Regional,2026-09-22,"1,935","45,789.60",45.7896,"23,481,846,730","9,857,444,439",41.98%,300,https://web.ksei.co.id/services/registered-securities/shares/lc/MEGA
7,8,BNGA,PT Bank CIMB Niaga Tbk,Financial Services,Banks - Regional,2026-09-22,"1,760","43,872.56",43.8726,"71,853,936","71,905,748",100.07%,"310,100",https://web.ksei.co.id/services/registered-securities/shares/lc/BNGA
8,9,BDMN,PT Bank Danamon Indonesia Tbk,Financial Services,Banks - Regional,2026-09-22,"4,450","43,492.31",43.4923,"22,400,000","707,898,434","3,160.26%","9,200",https://web.ksei.co.id/services/registered-securities/shares/lc/BDMN
9,10,NISP,PT Bank OCBC NISP Tbk,Financial Services,Banks - Regional,2026-09-22,"1,260","29,025.80",29.0258,"22,945,296,972","3,418,390,343",14.90%,"399,300",https://web.ksei.co.id/services/registered-securities/shares/lc/NISP


### Thermal Coal  
33 stocks; combined reported market cap: Rp1,165.76 trillion

,rank_in_industry,ticker,company_name,sector,industry_group,price_date,close_idr,market_cap_idr_billion,market_cap_idr_trillion,sharesOutstanding,floatShares,float_pct_of_outstanding,volume_shares,detail_url
0,1,BYAN,PT Bayan Resources Tbk.,Energy,Thermal Coal,2026-09-22,"13,400","440,000.01",440.0000,"33,333,335,000","7,092,667,021",21.28%,"1,478,700",https://web.ksei.co.id/services/registered-securities/shares/lc/BYAN
1,2,DSSA,PT Dian Swastatika Sentosa Tbk,Energy,Thermal Coal,2026-09-22,"1,005","156,279.43",156.2794,"154,732,123,250","39,343,736,979",25.43%,"49,698,200",https://web.ksei.co.id/services/registered-securities/shares/lc/DSSA
2,3,CUAN,PT Petrindo Jaya Kreasi Tbk,Energy,Thermal Coal,2026-09-22,940,"104,531.25",104.5312,"112,399,199,400","21,965,051,547",19.54%,"29,630,700",https://web.ksei.co.id/services/registered-securities/shares/lc/CUAN
3,4,AADI,PT Adaro Andalan Indonesia Tbk,Energy,Thermal Coal,2026-09-22,"11,575","88,770.57",88.7706,"7,786,891,760","2,896,022,914",37.19%,"383,200",https://web.ksei.co.id/services/registered-securities/shares/lc/AADI
4,5,ADRO,PT Alamtri Resources Indonesia Tbk,Energy,Thermal Coal,2026-09-22,"2,630","75,457.29",75.4573,"28,800,494,200","10,597,141,841",36.80%,"2,439,500",https://web.ksei.co.id/services/registered-securities/shares/lc/ADRO
5,6,BUMI,PT Bumi Resources Tbk,Energy,Thermal Coal,2026-09-22,192,"70,553.72",70.5537,"371,335,392,068","180,056,818,260",48.49%,"321,489,400",https://web.ksei.co.id/services/registered-securities/shares/lc/BUMI
6,7,GEMS,PT Golden Energy Mines Tbk,Energy,Thermal Coal,2026-09-22,"7,375","42,941.18",42.9412,"5,882,353,000","611,176,477",10.39%,"4,600",https://web.ksei.co.id/services/registered-securities/shares/lc/GEMS
7,8,PTBA,PT Bukit Asam (Persero) Tbk,Energy,Thermal Coal,2026-09-22,"3,050","34,658.22",34.6582,"11,514,357,250","3,776,939,465",32.80%,"1,483,500",https://web.ksei.co.id/services/registered-securities/shares/lc/PTBA
8,9,ITMG,PT Indo Tambangraya Megah Tbk,Energy,Thermal Coal,2026-09-22,"26,150","29,098.86",29.0989,"1,114,899,100","365,258,447",32.76%,"80,300",https://web.ksei.co.id/services/registered-securities/shares/lc/ITMG
9,10,BSSR,PT Baramulti Suksessarana Tbk,Energy,Thermal Coal,2026-09-22,"5,350","14,390.75",14.3908,"2,616,500,000","242,261,735",9.26%,"4,507,600",https://web.ksei.co.id/services/registered-securities/shares/lc/BSSR


### Telecom Services  
20 stocks; combined reported market cap: Rp860.57 trillion

...

,rank_in_industry,ticker,company_name,sector,industry_group,price_date,close_idr,market_cap_idr_billion,market_cap_idr_trillion,sharesOutstanding,floatShares,float_pct_of_outstanding,volume_shares,detail_url
0,1,TMPO,PT Tempo Inti Media Tbk,Communication Services,Publishing,2026-09-22,152,158.75,0.1587,"1,058,333,250","355,229,555",33.56%,"486,300",https://web.ksei.co.id/services/registered-securities/shares/lc/TMPO
1,2,ABBA,PT Mahaka Media Tbk,Communication Services,Publishing,2026-09-21,30,118.08,0.1181,"3,935,892,857","1,517,562,209",38.56%,0,https://web.ksei.co.id/services/registered-securities/shares/lc/ABBA


### Scientific & Technical Instruments  
1 stocks; combined reported market cap: Rp0.20 trillion

,rank_in_industry,ticker,company_name,sector,industry_group,price_date,close_idr,market_cap_idr_billion,market_cap_idr_trillion,sharesOutstanding,floatShares,float_pct_of_outstanding,volume_shares,detail_url
0,1,GPSO,PT Geoprima Solusi Tbk,Technology,Scientific & Technical Instruments,2026-09-22,264,195.09,0.1951,"733,415,103","385,129,663",52.51%,"14,867,600",https://web.ksei.co.id/services/registered-securities/shares/lc/GPSO


### Footwear & Accessories  
2 stocks; combined reported market cap: Rp0.13 trillion

,rank_in_industry,ticker,company_name,sector,industry_group,price_date,close_idr,market_cap_idr_billion,market_cap_idr_trillion,sharesOutstanding,floatShares,float_pct_of_outstanding,volume_shares,detail_url
0,1,BATA,PT Sepatu Bata Tbk.,Consumer Cyclical,Footwear & Accessories,2026-09-21,59,76.70,0.0767,"1,300,000,000","162,058,000",12.47%,0,https://web.ksei.co.id/services/registered-securities/shares/lc/BATA
1,2,BIMA,"PT. Primarindo Asia Infrastructure, Tbk.",Consumer Cyclical,Footwear & Accessories,2026-09-21,82,49.87,0.0499,"608,175,716","81,702,326",13.43%,0,https://web.ksei.co.id/services/registered-securities/shares/lc/BIMA


## Stocks that could not enter an industry table

In [7]:
# Objective: retain price-coverage failures as an explicit audit table.
exception_columns = ['ticker', 'yahoo_symbol', 'price_status', 'error']
missing_price_metadata = prices.loc[
    prices['price_status'].ne('available')
].reindex(columns=exception_columns).merge(
    registry[['ticker', 'registry_name', 'detail_url']],
    on='ticker',
    how='left',
    validate='one_to_one',
).sort_values('ticker')

missing_price_metadata

,ticker,yahoo_symbol,price_status,error,registry_name,detail_url
0,ARMY,ARMY.JK,missing,NaN,ARMIDIAN KARYATAMA Tbk,https://web.ksei.co.id/services/registered-sec...
1,ATPK,ATPK.JK,missing,NaN,BARA JAYA INTERNASIONAL Tbk,https://web.ksei.co.id/services/registered-sec...
2,BRAU,BRAU.JK,missing,NaN,BERAU COAL ENERGY Tbk,https://web.ksei.co.id/services/registered-sec...
3,BTEL,BTEL.JK,missing,NaN,BAKRIE TELECOM Tbk,https://web.ksei.co.id/services/registered-sec...
4,CBMF,CBMF.JK,missing,NaN,CAHAYA BINTANG MEDAN Tbk,https://web.ksei.co.id/services/registered-sec...
5,COWL,COWL.JK,missing,NaN,COWELL DEVELOPMENT Tbk,https://web.ksei.co.id/services/registered-sec...
6,CPRI,CPRI.JK,missing,NaN,CAPRI NUSA SATU PROPERTI Tbk,https://web.ksei.co.id/services/registered-sec...
7,DMAD,DMAD.JK,missing,NaN,DEEMADE KARYA INDONESIA Tbk,https://web.ksei.co.id/services/registered-sec...
8,DUCK,DUCK.JK,missing,NaN,JAYA BERSAMA INDO Tbk,https://web.ksei.co.id/services/registered-sec...
9,ENVY,ENVY.JK,missing,NaN,ENVY TECHNOLOGIES INDONESIA Tbk,https://web.ksei.co.id/services/registered-sec...


## Interpretation and refresh rules

- These tables organize a dated research snapshot; they are not recommendations.
- Yahoo industry labels are not IDX-IC. Do not describe them as official exchange sectors or industries.
- A large market capitalization does not imply good value, high liquidity, good governance, or low risk.
- A small nominal share price does not mean that a stock is cheap.
- Provider-reported market capitalization can use share counts and prices from different effective times.
- Verify material figures against issuer financial statements, IDX disclosures, and official share-registration information.
- Missing-price candidates remain visible in the exception table and are not silently discarded.
- Historical analysis requires point-in-time membership, delistings, suspensions, corporate actions, costs, taxes, spreads, and publication dates.

Rebuild this notebook after refreshing the dated KSEI, price, and metadata snapshots:

```bash
python3 scripts/build_stock_industry_tables_notebook.py
```

The notebook deliberately reads Yahoo bulk snapshots from `private/`; it will not execute on a fresh public clone until the user creates those snapshots with the repository's fetch scripts.